In [5]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import StringIO
import time

## Multiple Stations, Multiple Dates

In [6]:
# Create a list of the weather stations and the dates that we want to select
station_names = ['ILUTON8', 'ILUTON18', 'ILUTON30', 'ILUTON45', 'ILUTON47', 'ILUTON48', 'ILUTON49', 'ILUTON52', 'ILUTON55', 'ILUTON59', 'ILUTON62', 'ILUTON66']
selected_month = "June 2026"
#dates         = ['2026-06-26'] #  2026-06-26 2026-07-29
dates         = ['2026-06-01', '2026-06-02', '2026-06-03', '2026-06-04', '2026-06-05', '2026-06-06', '2026-06-07', '2026-06-08', '2026-06-09', '2026-06-10', 
                '2026-06-11', '2026-06-12', '2026-06-13', '2026-06-14', '2026-06-15', '2026-06-16', '2026-06-17', '2026-06-18', '2026-06-19', '2026-06-20',
                '2026-06-21', '2026-06-22', '2026-06-23', '2026-06-24', '2026-06-25', '2026-06-26', '2026-06-27', '2026-06-28', '2026-06-29', '2026-06-30']

In [7]:
def scrape_wunderground_multiple(df_master, station_list, date_list, target_month):
    
    """
    Function to scrape hourly data from multiple stations on multiple dates.
    """
    
    # For every station
    for station in station_list:
        
        # For every date
        for date in date_list:
            
            # Set the URL
            url = 'https://preview.wunderground.com/dashboard/pws/' + station
            
            # Open in chrome
            driver = webdriver.Chrome()
            driver.get(url)
            
            wait = WebDriverWait(driver, 30)

            # Pause for 10 seconds to allow cookie pop-up to open
            time.sleep(7)
            
            # Find the number of frames and switch to the one with the popup
            iframes = driver.find_elements(By.TAG_NAME, "iframe")
            driver.switch_to.frame(iframes[1])

            # Click accept all cookies and then go back to default frame
            accept_button = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'button[aria-label="Accept all"]')
                )
            )
             
            accept_button.click()
             
            driver.switch_to.default_content()

            # Let the page load for 10 seconds
            time.sleep(10)

            # Scroll down to the custom date button
            element = driver.find_element(
                By.CSS_SELECTOR,
                    '[data-range="custom"]'
                )
             
            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                    element
                )     


            # Click the custom date button
            custom_button = wait.until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, '[data-range="custom"]')
                )
            )
             
            custom_button.click()

            time.sleep(2)
            # Select a target month and then go back in the calendar until you reach it
             
            while True:
             
                current_month = driver.find_element(
                    By.CSS_SELECTOR,
                    "span.calendar-title"
                ).text
             
                #print(current_month)
             
                if current_month == target_month:
                    break
             
                driver.find_element(
                    By.CSS_SELECTOR,
                        "button.calendar-prev"
                    ).click()


            # Select start date
            #elemt_date = "[data-date="2026-""
            driver.find_element(
                By.CSS_SELECTOR,
                    '[data-date="' + date + '"]'
                ).click()
             
            # Select end date
            driver.find_element(
                By.CSS_SELECTOR,
                    '[data-date="' + date + '"]'
                ).click()


            time.sleep(3)
            
            # Click the table button (originally "graph" is selected)
            table_button = wait.until(
                EC.element_to_be_clickable(
                    (By.CSS_SELECTOR, 'button[data-view="table"]')
                )
            )
            
            table_button.click()


            time.sleep(2)
            
            # Get the tables on the webpage
            tables = WebDriverWait(driver,20).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table")))
            
            # Extract HTML
            html = tables[1].get_attribute("outerHTML")
            # Convert to DataFrame
            df = pd.read_html(StringIO(html))[0]
            
            # Set a new name for each station + date
            df['station_id'] = station
            df['date'] = date

            df_master = pd.concat([df_master, df])

            driver.quit()

    return df_master

In [8]:
%%time
df_Luton = pd.DataFrame()
df_Luton = scrape_wunderground_multiple(df_Luton, station_names, dates, selected_month)

CPU times: total: 57.3 s
Wall time: 3h 12min 8s


In [10]:
%%time
df_Luton.to_csv("Luton_wunderground_stations_all_06_jun.csv")

CPU times: total: 156 ms
Wall time: 174 ms
